# Giai đoạn 0
### Khảo sát và Hiểu Bộ Dữ liệu CWRU

**Mục lục:**
1. [01 — Dataset Overview](#01) (mục 0.1)
2. [02 — Time Domain Analysis](#02) (mục 0.2a)
3. [03 — Frequency Domain Analysis](#03) (mục 0.2b)
4. [04 — Order-Domain Analysis](#04) (mục 1.2)
5. [05 — Envelope Analysis](#05) (mục 1.3)
6. [06 — Window Leakage Analysis](#06) (mục 0.3a)
7. [07 — LOLO Visualization](#07) (mục 0.3b/2.1)
8. [08 — Summary & Research Questions](#08) (mục 0.4)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from typing import Any
from matplotlib.patches import Rectangle

from common import io_utils, pipeline, synthetic, dsp, features, splitting, config as cfg

pd.set_option("display.max_colwidth", 120)

In [2]:
# ============================== CẤU HÌNH ==============================
# Đổi USE_SYNTHETIC_DATA = False và chỉnh REAL_DATA_ROOT khi đã có dữ liệu
# CWRU thật. Xem README.md phần "Chuyển sang dữ liệu thật".
USE_SYNTHETIC_DATA = False
REAL_DATA_ROOT = Path("../../data/raw")            # <-- data/raw/
SYNTHETIC_DATA_ROOT = Path("./_data/synthetic_cwru")
OUTPUT_DIR = Path("./outputs")
FORCE_REBUILD_MANIFEST = True
# ========================================================================

In [3]:
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

<a id="01"></a>

## 01 — Dataset Overview
### Mục 0.1 của đề cương — Khảo sát cấu trúc bộ dữ liệu CWRU

**Mục đích**: quét thư mục dữ liệu, dựng bảng manifest (nhãn, tải, đường
kính lỗi, vị trí cảm biến...), và chạy các sanity check quan trọng đã xác
định ở phần Tổng quan đề tài:

1. Sampling rate của file Normal baseline (nhiều nguồn không thống nhất
   12kHz hay 48kHz).
2. Đường kính lỗi 0.028"/0.040" dùng vòng bi **NTN**, khác SKF 6205 dùng
   cho 0.007"/0.014"/0.021".
3. Outer Race có 3 vị trí lỗi (6h/3h/12h) — đề tài chỉ dùng vị trí
   **Centered (6h)**.
4. RPM thực tế trong file có khớp RPM danh định theo tải hay không.

**Trước khi chạy với dữ liệu thật**: mở `common/io_utils.py`, chỉnh hàm
`parse_metadata_from_filename()` cho khớp cấu trúc thư mục/tên file bạn
đang có — file `.mat` gốc CWRU chỉ có tên số, không tự chứa metadata.

In [4]:
manifest = pipeline.get_manifest(
    use_synthetic=USE_SYNTHETIC_DATA,
    real_data_root=REAL_DATA_ROOT,
    synthetic_data_root=SYNTHETIC_DATA_ROOT,
    output_dir=OUTPUT_DIR,
    force_rebuild=FORCE_REBUILD_MANIFEST,
)
print(f"Tổng số file trong manifest: {len(manifest)}")
manifest.head()

Tổng số file trong manifest: 161


,file_path,load_hp,label,fault_diameter_mils,or_position,source_category,sensor_location,declared_sample_rate_khz,n_samples_DE,n_samples_FE,n_samples_BA,rpm_from_file,read_error,warnings,has_warning
0,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\007\118_0.mat,0,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,122571,122571.0,122571.0,1796.0,None,,False
1,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\007\119_1.mat,1,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121410,121410.0,121410.0,1772.0,None,,False
2,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\007\120_2.mat,2,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1748.0,None,,False
3,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\007\121_3.mat,3,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1722.0,None,,False
4,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\014\185_0.mat,0,B,14.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121846,121846.0,121846.0,1796.0,None,,False


## Thống kê số file theo (nhãn × tải)

In [5]:
pivot = manifest.pivot_table(
    index="label", columns="load_hp", values="file_path",
    aggfunc="count", fill_value=0,
)
print(pivot)
pivot.to_csv(TABLES_DIR / "01_label_load_pivot.csv")

load_hp   0   1   2   3
label                  
B        10  10  10  10
IR       10  10  10  10
Normal    1   1   1   1
OR       20  19  19  19


## Cảnh báo sanity check

Mỗi dòng cảnh báo cần được đọc và quyết định xử lý (giữ/loại/sửa) — script chỉ phát hiện

In [6]:
n_warn = int(manifest["has_warning"].sum())
print(f"Số file có cảnh báo: {n_warn} / {len(manifest)}\n")

if n_warn > 0:
    warn_cols = ["file_path", "label", "load_hp", "fault_diameter_mils", "warnings"]
    warn_df = manifest.loc[manifest["has_warning"], warn_cols]
    warn_df.to_csv(TABLES_DIR / "01_warnings.csv", index=False)
    display(warn_df)
else:
    print("Không có cảnh báo nào trên bộ dữ liệu hiện tại.")

Số file có cảnh báo: 125 / 161



,file_path,label,load_hp,fault_diameter_mils,warnings
12,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\028\3005_0.mat,B,0,28.0,"VONG_BI_NTN: đường kính 28 mils dùng vòng bi NTN, KHÔNG dùng hình học SKF 6205 để tính BPFO/BPFI/BSF."
13,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\028\3006_1.mat,B,1,28.0,"VONG_BI_NTN: đường kính 28 mils dùng vòng bi NTN, KHÔNG dùng hình học SKF 6205 để tính BPFO/BPFI/BSF."
14,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\028\3007_2.mat,B,2,28.0,"VONG_BI_NTN: đường kính 28 mils dùng vòng bi NTN, KHÔNG dùng hình học SKF 6205 để tính BPFO/BPFI/BSF."
15,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\B\028\3008_3.mat,B,3,28.0,"VONG_BI_NTN: đường kính 28 mils dùng vòng bi NTN, KHÔNG dùng hình học SKF 6205 để tính BPFO/BPFI/BSF."
28,..\..\data\raw\12k_Drive_End_Bearing_Fault_Data\IR\028\3001_0.mat,IR,0,28.0,"VONG_BI_NTN: đường kính 28 mils dùng vòng bi NTN, KHÔNG dùng hình học SKF 6205 để tính BPFO/BPFI/BSF."
...,...,...,...,...,...
156,..\..\data\raw\48k_Drive_End_Bearing_Fault_Data\OR\021\@6\241_3.mat,OR,3,21.0,"NGOAI_PHAM_VI_TAN_SO_KHAI_BAO: file thuộc nhóm khai báo 48.0kHz, phạm vi đã chốt chỉ dùng 12kHz."
157,..\..\data\raw\Normal\100_Normal_3.mat,Normal,3,NaN,"NGHI_NGO_SAMPLING_RATE: n_samples=485643 (12kHz->40.5s, 24kHz->20.2s, 48kHz->10.1s). Rate hợp lý nhất: 48kHz — KHÁC ..."
158,..\..\data\raw\Normal\97_Normal_0.mat,Normal,0,NaN,"NGHI_NGO_SAMPLING_RATE: n_samples=243938 (12kHz->20.3s, 24kHz->10.2s, 48kHz->5.1s). Rate hợp lý nhất: 24kHz — KHÁC r..."
159,..\..\data\raw\Normal\98_Normal_1.mat,Normal,1,NaN,"NGHI_NGO_SAMPLING_RATE: n_samples=483903 (12kHz->40.3s, 24kHz->20.2s, 48kHz->10.1s). Rate hợp lý nhất: 48kHz — KHÁC ..."


## Tự kiểm chứng sanity check (bonus)

Phần dưới đây tạo riêng 4 file `.mat` **cố ý gài lỗi** (tách biệt hoàn
toàn khỏi `manifest` ở trên) để xác nhận cả 4 loại sanity check thực sự
hoạt động, trước khi bạn tin tưởng dùng chúng trên dữ liệu thật.

In [7]:
edge_root, expected_keywords = synthetic.build_edge_case_dataset(Path("./_data/edge_cases"))
edge_manifest = io_utils.build_manifest(edge_root)

all_warnings_text = " ".join(edge_manifest["warnings"].tolist())
print("Kiểm tra từng loại cảnh báo có xuất hiện không:\n")
all_ok = True
for kw in expected_keywords:
    found = kw in all_warnings_text
    print(f"  [{'CÓ' if found else 'THIẾU'}] {kw}")
    all_ok = all_ok and found

print("\nToàn bộ sanity check hoạt động đúng." if all_ok else
      "\nCó sanity check chưa hoạt động như mong đợi — kiểm tra lại common/io_utils.py.")

Kiểm tra từng loại cảnh báo có xuất hiện không:

  [CÓ] NGHI_NGO_SAMPLING_RATE
  [CÓ] OR_NGOAI_PHAM_VI
  [CÓ] VONG_BI_NTN
  [CÓ] RPM_LECH
  [CÓ] NGOAI_PHAM_VI_CAM_BIEN
  [CÓ] NGOAI_PHAM_VI_TAN_SO_KHAI_BAO

Toàn bộ sanity check hoạt động đúng.


## Checklist trước khi sang notebook 02

- [ ] Đã chỉnh `parse_metadata_from_filename()` khớp với dữ liệu thật.
- [ ] Đã đọc hết `outputs/tables/01_warnings.csv`, quyết định rõ giữ/loại
      từng trường hợp.
- [ ] Đã đối chiếu ngẫu nhiên vài dòng manifest với bảng tra cứu gốc trên
      trang CWRU Bearing Data Center để xác nhận parse đúng.